# Event History Analysis (EHA)

## Supervivencia de gabinetes

Este notebook comienza con una base ya preparada por **preEHA**.

Aquí no limpiamos fechas ni construimos las variables de duración y evento.
Nos concentramos en **describir, comparar y modelar la supervivencia de los gabinetes**.

## EHA-01 — Leer la base preparada

In [ ]:
import pandas as pd

# Sustituir por la URL publicada de la salida de preEHA.
url_eha = "PEGAR_AQUI_URL_PICKLE_PROCESADO"

eha = pd.read_pickle(url_eha)
eha.head()

## EHA-02 — Reconocer eventos y casos censurados

In [ ]:
resumen_eventos = (
    eha["caida_evento"]
    .value_counts()
    .rename(index={1: "Caída observada", 0: "Censurado"})
)

print(resumen_eventos)

### Qué leer

`caida_evento = 1` significa que observamos la caída del gabinete.

`caida_evento = 0` significa que el gabinete seguía vigente cuando terminó su observación.
No lo eliminamos: sabemos **cuánto tiempo logró sobrevivir**, aunque no observamos su duración final.

## EHA-03 — Estimar la supervivencia de todos los gabinetes

In [ ]:
!pip install lifelines --quiet

import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

km = KaplanMeierFitter()

km.fit(
    durations=eha["duracion_meses"],
    event_observed=eha["caida_evento"],
    label="Gabinetes"
)

km.plot_survival_function()
plt.xlabel("Meses")
plt.ylabel("Probabilidad estimada de supervivencia")
plt.title("Supervivencia de los gabinetes")
plt.show()

### Cómo leer la curva

Comenzamos cerca de 1 porque, al inicio, todos los gabinetes están en funciones.

La curva desciende cuando observamos caídas.

En cualquier momento podemos preguntar:

> ¿Qué proporción estimada de gabinetes continúa en funciones más allá de este tiempo?

## EHA-04 — Comparar gobiernos mayoritarios y minoritarios

In [ ]:
km_mayoria = KaplanMeierFitter()
km_minoria = KaplanMeierFitter()

mayoria = eha["gobierno_mayoritario"] == 1

ax = km_mayoria.fit(
    eha.loc[mayoria, "duracion_meses"],
    eha.loc[mayoria, "caida_evento"],
    label="Mayoritario"
).plot_survival_function()

km_minoria.fit(
    eha.loc[~mayoria, "duracion_meses"],
    eha.loc[~mayoria, "caida_evento"],
    label="Minoritario"
).plot_survival_function(ax=ax)

plt.xlabel("Meses")
plt.ylabel("Probabilidad estimada de supervivencia")
plt.title("Supervivencia según tipo de gobierno")
plt.show()

### Qué comparar

Si una curva permanece por encima de la otra, ese grupo muestra mayor supervivencia durante esa parte del período observado.

Primero miramos el patrón. Después evaluamos si la diferencia es suficientemente sistemática.

## EHA-05 — Evaluar la diferencia entre las curvas

In [ ]:
from lifelines.statistics import logrank_test

logrank = logrank_test(
    eha.loc[mayoria, "duracion_meses"],
    eha.loc[~mayoria, "duracion_meses"],
    event_observed_A=eha.loc[mayoria, "caida_evento"],
    event_observed_B=eha.loc[~mayoria, "caida_evento"]
)

print("p-valor Log-Rank:", round(logrank.p_value, 4))

### Cómo leerlo

El Log-Rank pregunta si la experiencia de supervivencia de ambos grupos puede considerarse igual a lo largo del tiempo.

El p-valor no nos dice **cuánto** difieren políticamente. Para eso seguimos mirando las curvas y luego modelamos el riesgo.

## EHA-06 — Modelar el riesgo de caída

In [ ]:
from lifelines import CoxPHFitter

eha_cox = eha[
    [
        "duracion_meses",
        "caida_evento",
        "gobierno_mayoritario",
        "fragmentacion",
        "crecimiento_pbi"
    ]
].copy()

cox = CoxPHFitter()
cox.fit(
    eha_cox,
    duration_col="duracion_meses",
    event_col="caida_evento"
)

cox.print_summary()

### Qué leer primero

En la tabla de Cox, concentre la atención en `exp(coef)`.

Ese valor es el **Hazard Ratio (HR)**:

- `HR = 1`: no cambia el riesgo relativo;
- `HR > 1`: mayor riesgo instantáneo de caída;
- `HR < 1`: menor riesgo instantáneo de caída.

No confunda el HR con una probabilidad de supervivencia ni con meses de duración.

## EHA-07 — Extraer los Hazard Ratios

In [ ]:
resultado_hr = pd.DataFrame({
    "HR": cox.hazard_ratios_,
    "p_valor": cox.summary["p"]
}).round(3)

resultado_hr

### La pregunta vuelve a ser política

Para `gobierno_mayoritario`, un HR menor que 1 indica que los gabinetes mayoritarios presentan menor riesgo instantáneo de caída que los minoritarios, manteniendo constantes las otras variables del modelo.

Para `fragmentacion`, un HR mayor que 1 indica que mayores niveles de fragmentación están asociados con mayor riesgo instantáneo de caída.

## EHA-08 — Comprobar el supuesto de riesgos proporcionales

In [ ]:
cox.check_assumptions(
    eha_cox,
    p_value_threshold=0.05,
    show_plots=False
)

### Por qué hacemos este diagnóstico

El modelo de Cox supone que la relación entre los riesgos permanece proporcional a lo largo del tiempo.

No necesitamos memorizar el procedimiento de diagnóstico. Necesitamos saber qué pregunta responde:

> ¿Es razonable utilizar un único Hazard Ratio para resumir la relación durante todo el período observado?

Si el diagnóstico señala problemas, debemos revisar esa interpretación.

## EHA-09 — Reconstruir la evidencia

In [ ]:
print("LOG-RANK")
print("p-valor:", round(logrank.p_value, 4))

print("\nHAZARD RATIOS")
display(resultado_hr)

## Cierre

Al terminar debemos poder responder:

1. ¿Cuál es el evento?
2. ¿Qué significa un caso censurado?
3. ¿Qué muestra una curva Kaplan-Meier?
4. ¿Qué grupo presenta mayor supervivencia?
5. ¿Qué aporta el Log-Rank?
6. ¿Qué significa un Hazard Ratio menor o mayor que 1?
7. ¿Qué nos dicen mayoría parlamentaria y fragmentación?
8. ¿El supuesto de riesgos proporcionales parece razonable?

La preparación pertenece a **preEHA**. Este notebook comienza con la base lista para survival analysis.